# EX_05 — Vector stores y retrieval (ejercicios)

**Notebook de referencia:** `notebook/05_Vectorstores_Retrieval.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Chunking

Implementa un chunker trivial por **número de caracteres** con solapamiento (`chunk_size`, `chunk_overlap`). Aplícalo a un texto largo en una lista de strings.


In [1]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    # Control de seguridad para evitar bucles infinitos si los parámetros están mal configurados
    if chunk_size <= 0:
        raise ValueError("chunk_size debe ser mayor que 0.")
    if overlap >= chunk_size:
        raise ValueError("El overlap no puede ser mayor o igual que el chunk_size.")
        
    chunks = []
    start = 0
    text_len = len(text)
    
    # Recorremos el texto mientras el índice de inicio no haya llegado al final
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        
        # El truco: avanzamos el tamaño del bloque MENOS el solapamiento
        start += (chunk_size - overlap)
        
        # Condición de salida si el último bloque ya alcanzó o pasó el final del texto
        if end >= text_len:
            break
            
    return chunks

long_text = "word " * 500  # Genera un string largo de prueba (2500 caracteres)

resultado_chunks = chunk_text(long_text, chunk_size=200, overlap=40)

print(f"Longitud total del texto original: {len(long_text)} caracteres.")
print(f"Número total de chunks generados: {len(resultado_chunks)}")
print(f"Longitud del primer chunk: {len(resultado_chunks[0])} caracteres.")

Longitud total del texto original: 2500 caracteres.
Número total de chunks generados: 16
Longitud del primer chunk: 200 caracteres.


## Actividad 2 — Embeddings + FAISS

Embedde los chunks (puede ser `sentence_transformers`) y construye un índice `faiss.IndexFlatIP` o `IndexFlatL2`. Recupera los top-3 para una query.

*Hint:* L2-normalize vectors if you treat inner product as cosine similarity.


In [2]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Preparar el modelo y los datos (usando chunks de la actividad anterior)
model = SentenceTransformer('all-MiniLM-L6-v2')
chunks = ["El sistema solar tiene ocho planetas.", 
          "La inteligencia artificial transforma la industria.", 
          "Plutón ya no es considerado un planeta principal.",
          "Los modelos de lenguaje usan redes neuronales.",
          "Júpiter es el planeta más grande del sistema."]

# 2. Generar embeddings y convertirlos a float32 (requisito de FAISS)
embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype('float32')

# 3. Normalizar los vectores (L2 normalization) para usar Inner Product como Coseno
faiss.normalize_L2(embeddings)

# 4. Construir el índice FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # IndexFlatIP + vectores normalizados = Cosine Similarity
index.add(embeddings)

# 5. Realizar la búsqueda (Query)
query = "Cuéntame cosas sobre los planetas"
query_embedding = model.encode([query]).astype('float32')
faiss.normalize_L2(query_embedding)

# Recuperar Top-3
k = 3
distances, indices = index.search(query_embedding, k)

# 6. Mostrar resultados
print(f"Resultados para: '{query}'\n")
for i in range(k):
    idx = indices[0][i]
    score = distances[0][i]
    print(f"Top {i+1}: {chunks[idx]} (Score: {score:.4f})")


c:\Users\ignar\iniciacion_NLP_LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2055.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resultados para: 'Cuéntame cosas sobre los planetas'

Top 1: Júpiter es el planeta más grande del sistema. (Score: 0.6985)
Top 2: Plutón ya no es considerado un planeta principal. (Score: 0.6650)
Top 3: El sistema solar tiene ocho planetas. (Score: 0.6148)


## Actividad 3 — Métrica manual

Para una query y tres documentos **artificiales** (uno relevante, dos ruido), muestra scores de similitud y verifica que el relevante queda primero.


In [4]:
# TODO: synthetic docs + cosine sim ranking
from sentence_transformers import SentenceTransformer, util

# 1. Inicializamos el modelo de embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Definimos la Query y los Documentos Sintéticos
query = "How do transformers utilize self-attention?"

docs = [
    "Noise Doc A: Attention mechanisms are also used in traditional computer vision architectures like ResNet.", # Ruido (misma palabra, contexto equivocado)
    "Noise Doc B: Baking cookies requires a stable oven temperature and high-quality chocolate chips.",         # Ruido absoluto
    "Relevant Doc: The self-attention layers in Transformers allow tokens to dynamically weight historical context." # El relevante
]

# 3. Generamos los embeddings para la query y todos los documentos
query_emb = model.encode(query)
docs_embs = model.encode(docs)

# 4. Calculamos los scores de similitud coseno de forma manual/directa
# util.cos_sim nos devolverá una matriz de similitudes
scores = util.cos_sim(query_emb, docs_embs)[0].tolist()

# 5. Emparejamos cada documento con su score y los ordenamos de mayor a menor
ranked_results = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

# 6. Imprimimos el ranking resultante
print("--- RANKING DE SIMILITUD COSENO ---")
for rank, (doc, score) in enumerate(ranked_results, 1):
    print(f"Puesto {rank}: Score = {score:.4f} -> {doc}")

# 7. Verificación automática: El primer documento debe ser el relevante
assert "Relevant Doc" in ranked_results[0][0], "¡Error! El documento relevante no quedó primero."
print("\n¡Verificación exitosa! El documento relevante lidera el ranking.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6843.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- RANKING DE SIMILITUD COSENO ---
Puesto 1: Score = 0.5946 -> Relevant Doc: The self-attention layers in Transformers allow tokens to dynamically weight historical context.
Puesto 2: Score = 0.4498 -> Noise Doc A: Attention mechanisms are also used in traditional computer vision architectures like ResNet.
Puesto 3: Score = -0.0216 -> Noise Doc B: Baking cookies requires a stable oven temperature and high-quality chocolate chips.

¡Verificación exitosa! El documento relevante lidera el ranking.
